
* ###  PREPARA ARCHIVO DETALLADO DE LOS PAGOS REALIZADOS Y ENVIA A BORRADORES DETALLE Y COMPROBANTE 

1️⃣ SCRIPT 1 — SOLO CREAR ARCHIVO FILTRADO

In [ ]:
import pandas as pd
from datetime import datetime

# =====================================================
# CONFIGURACIÓN
# =====================================================

archivo_excel = 'Recepción.xlsx'
hoja = 'REGISTROS'
columna_fecha = 'Fecha Pago Cargado Tesoreria'

columnas_deseadas = [
    'Empresa', 'Proveedor', 'Factura', 'Monto', 'Moneda', 'Fecha Pago Cargado Tesoreria'
]

empresa_filtrar = 'Nutrex_INC'
archivo_salida = 'datos_filtrados_nutrex.xlsx'

# =====================================================
# FILTRAR PAGOS
# =====================================================

try:

    df = pd.read_excel(archivo_excel, sheet_name=hoja)

    # limpiar nombres de columnas
    df.columns = df.columns.str.strip()

    df = df[df['Empresa'] == empresa_filtrar]

    df[columna_fecha] = pd.to_datetime(df[columna_fecha], errors='coerce')

    fechas_input = input(
        "\n👉 Ingresa fecha(s) de pago (YYYY-MM-DD), separadas por coma, "
        "o deja vacío para usar la más reciente: "
    ).strip()

    if fechas_input == '':
        fechas_filtrar = [df[columna_fecha].max().date()]
        print(f"\n📌 Se usará la fecha más reciente: {fechas_filtrar[0]}")
    else:
        fechas_filtrar = [
            datetime.strptime(f.strip(), '%Y-%m-%d').date()
            for f in fechas_input.split(',')
        ]

    df_filtrado = df[df[columna_fecha].dt.date.isin(fechas_filtrar)]

    if df_filtrado.empty:
        print("❌ No hay datos para las fechas indicadas.")
        raise SystemExit()

    df_filtrado = df_filtrado[columnas_deseadas]

    df_filtrado.to_excel(archivo_salida, index=False)

    print(f"\n✅ Archivo generado: {archivo_salida}")

except Exception as e:
    print("\n❌ ERROR:")
    print(e)

C:\Users\Nutrex\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)



✅ Archivo generado: datos_filtrados_nutrex.xlsx


2️⃣ SCRIPT 2 — CREAR CORREOS EN BORRADORES

In [4]:
import pandas as pd
import os
import win32com.client as win32

# =====================================================
# CONFIGURACIÓN
# =====================================================

archivo_datos = 'datos_filtrados_nutrex.xlsx'
archivo_remitentes = 'remitentes.xlsx'

columna_proveedor = 'Proveedor'
columna_fecha = 'Fecha Pago Cargado Tesoreria'
columna_para = 'PARA'
columna_cc = 'CC'

ruta_adjuntos = r'C:\Users\Nutrex\OneDrive - NUTREX PARAGUAY SRL\Documentos - Documentos\ADMINISTRACION\25.BANCOS\Bancos BVI\Bancos Actuales\1-Bancolombia\Transferencias\2026'

# =====================================================
# CREAR CORREOS
# =====================================================

try:

    df = pd.read_excel(archivo_datos)
    remitentes_df = pd.read_excel(archivo_remitentes)

    df.columns = df.columns.str.strip()
    remitentes_df.columns = remitentes_df.columns.str.strip().str.upper()

    df[columna_proveedor] = df[columna_proveedor].astype(str).str.strip().str.upper()
    remitentes_df['PROVEEDOR'] = remitentes_df['PROVEEDOR'].astype(str).str.strip().str.upper()

    df = df.merge(
        remitentes_df,
        left_on=columna_proveedor,
        right_on='PROVEEDOR',
        how='left'
    )

    faltantes = df[df[columna_para].isna()][columna_proveedor].unique()

    if len(faltantes) > 0:
        print("❌ Proveedores sin correo:")
        for f in faltantes:
            print(f" - {f}")
        raise SystemExit()

    outlook = win32.Dispatch('Outlook.Application')

    if not os.path.exists(ruta_adjuntos):
        raise FileNotFoundError(f"No existe la carpeta: {ruta_adjuntos}")

    archivos_pdf = [
        f for f in os.listdir(ruta_adjuntos)
        if f.lower().endswith('.pdf') and not f.startswith('~')
    ]

    for (proveedor, fecha_pago), grupo in df.groupby([columna_proveedor, columna_fecha]):

        correo_para = grupo[columna_para].iloc[0]
        correo_cc = grupo[columna_cc].iloc[0] if columna_cc in grupo.columns else ''

        fecha_pdf = pd.to_datetime(fecha_pago).strftime('%Y%m%d')
        proveedor_key = proveedor.replace(' ', '').upper()

        adjunto_encontrado = None

        for archivo in archivos_pdf:

            archivo_key = archivo.replace(' ', '').upper()

            if archivo_key.startswith(fecha_pdf) and proveedor_key in archivo_key:

                adjunto_encontrado = os.path.join(ruta_adjuntos, archivo)
                break

        columnas_excluir = ['PROVEEDOR', columna_para, columna_cc]
        columnas_excluir = [c for c in columnas_excluir if c in grupo.columns]

        tabla_html = grupo.drop(columns=columnas_excluir).to_html(
            index=False,
            border=0,
            justify='center'
        )

        estilo_tabla = """
        <style>
        table {border-collapse: collapse; width:100%; font-family:Arial; font-size:13px;}
        th {background-color:#1f3a5f;color:white;padding:6px;border:1px solid #1f3a5f;text-align:center;}
        td {padding:6px;border:1px solid #1f3a5f;text-align:center;}
        </style>
        """

        cuerpo_html = f"""
        <html>
        <head>{estilo_tabla}</head>
        <body style="font-family:Arial;font-size:13px;">

        <p>Buenos días,</p>

        <p>Adjuntamos comprobante de pago correspondiente a las siguientes facturas:</p>

        {tabla_html}

        <br>

        <p>Ante cualquier duda, quedo a disposición.</p>
        <p>Buen resto de día,</p>
        <p>Saludos cordiales,</p>

        <p>
        <b>VICTOR DOMINGUEZ</b><br>
        Nutrex INC<br>
        tesoreria2@nutrexinc.com
        </p>

        </body>
        </html>
        """

        mail = outlook.CreateItem(0)

        mail.To = correo_para

        if correo_cc:
            mail.CC = correo_cc

        mail.Subject = f"COMPROBANTE DE PAGO {pd.to_datetime(fecha_pago).strftime('%d/%m/%Y')} - {proveedor}"

        mail.HTMLBody = cuerpo_html

        if adjunto_encontrado:
            mail.Attachments.Add(adjunto_encontrado)
            print(f"📎 PDF encontrado → {proveedor}")
        else:
            print(f"⚠️ PDF NO encontrado → {proveedor}")

        mail.Save()

        print(f"📧 Correo creado → {proveedor}")

    print("\n✅ Correos creados en borradores")

except Exception as e:
    print("\n❌ ERROR:")
    print(e)

⚠️ PDF NO encontrado → AGENCIA MARITIMA INTERNACIONAL S.A.
📧 Correo creado → AGENCIA MARITIMA INTERNACIONAL S.A.
⚠️ PDF NO encontrado → ARACELI DE JESUS SANABRIA CACERES
📧 Correo creado → ARACELI DE JESUS SANABRIA CACERES
⚠️ PDF NO encontrado → CARSTANJEN & PENTZ COMMERCIAL PARTNERS LLC
📧 Correo creado → CARSTANJEN & PENTZ COMMERCIAL PARTNERS LLC
⚠️ PDF NO encontrado → CEOLOGISTICS INTERNATIONAL LLC
📧 Correo creado → CEOLOGISTICS INTERNATIONAL LLC
⚠️ PDF NO encontrado → G3 TRANSPORTES S.A.
📧 Correo creado → G3 TRANSPORTES S.A.
⚠️ PDF NO encontrado → INPASA DEL PARAGUAY SA
📧 Correo creado → INPASA DEL PARAGUAY SA
⚠️ PDF NO encontrado → MAERSK LINE A/S
📧 Correo creado → MAERSK LINE A/S
⚠️ PDF NO encontrado → MSC BOLIVIA LTDA
📧 Correo creado → MSC BOLIVIA LTDA
⚠️ PDF NO encontrado → OCI CHEM LLC
📧 Correo creado → OCI CHEM LLC
⚠️ PDF NO encontrado → PUERTO SEGURO FLUVIAL S.A
📧 Correo creado → PUERTO SEGURO FLUVIAL S.A
⚠️ PDF NO encontrado → REDFLINT PARAGUAY EAS
📧 Correo creado → REDFLINT 

🖥️ SCRIPT CON VENTANA TKINTER — FILTRAR + CREAR CORREOS EN BORRADORES

In [ ]:
"""import pandas as pd
import os
import tkinter as tk
from tkinter import messagebox
from datetime import datetime
import win32com.client as win32

# =====================================================
# === CONFIGURACIÓN GENERAL ===
# =====================================================

archivo_excel = 'Recepción.xlsx'
hoja = 'REGISTROS'
columna_fecha = 'Fecha Pago Cargado Tesoreria'
columnas_deseadas = [
    'Empresa', 'Proveedor', 'Factura', 'Monto', 'Moneda', 'Fecha Pago Cargado Tesoreria'
]
empresa_filtrar = 'Nutrex_INC'
archivo_salida = 'datos_filtrados_nutrex.xlsx'

archivo_remitentes = 'remitentes.xlsx'
columna_proveedor = 'Proveedor'
columna_para = 'PARA'
columna_cc = 'CC'

ruta_adjuntos = r'C:\Users\Nutrex\OneDrive - NUTREX PARAGUAY SRL\Documentos - Documentos\ADMINISTRACION\25.BANCOS\Bancos BVI\Bancos Actuales\1-Bancolombia\Transferencias\2026'


# =====================================================
# === FUNCIONES ===
# =====================================================

def filtrar_y_enviar(fechas_texto):
    """Filtra datos por fecha y crea correos en borradores de Outlook."""

    # --- PARTE 1: FILTRAR ---
    try:
        df = pd.read_excel(archivo_excel, sheet_name=hoja)
        df.columns = df.columns.str.strip()
        df = df[df['Empresa'] == empresa_filtrar]
        df[columna_fecha] = pd.to_datetime(df[columna_fecha], errors='coerce')

        if fechas_texto.strip() == '':
            fechas_filtrar = [df[columna_fecha].max().date()]
        else:
            fechas_filtrar = [
                datetime.strptime(f.strip(), '%Y-%m-%d').date()
                for f in fechas_texto.split(',')
            ]

        df_filtrado = df[df[columna_fecha].dt.date.isin(fechas_filtrar)]

        if df_filtrado.empty:
            messagebox.showwarning("Sin datos", f"No hay datos para las fechas: {', '.join(str(f) for f in fechas_filtrar)}")
            return

        df_filtrado = df_filtrado[columnas_deseadas]
        df_filtrado.to_excel(archivo_salida, index=False)
        print(f"✅ Archivo filtrado generado: {archivo_salida}")
        print(f"📅 Fechas: {', '.join(str(f) for f in fechas_filtrar)}")

    except Exception as e:
        messagebox.showerror("Error en filtrado", str(e))
        return

    # --- PARTE 2: CREAR CORREOS ---
    try:
        df = pd.read_excel(archivo_salida)
        remitentes_df = pd.read_excel(archivo_remitentes)

        df.columns = df.columns.str.strip()
        remitentes_df.columns = remitentes_df.columns.str.strip().str.upper()

        df[columna_proveedor] = df[columna_proveedor].astype(str).str.strip().str.upper()
        remitentes_df['PROVEEDOR'] = remitentes_df['PROVEEDOR'].astype(str).str.strip().str.upper()

        df = df.merge(
            remitentes_df,
            left_on=columna_proveedor,
            right_on='PROVEEDOR',
            how='left'
        )

        faltantes = df[df[columna_para].isna()][columna_proveedor].unique()
        if len(faltantes) > 0:
            lista = '\n'.join(f' - {f}' for f in faltantes)
            messagebox.showerror("Proveedores sin correo", f"Faltan correos para:\n{lista}")
            return

        outlook = win32.Dispatch('Outlook.Application')

        if not os.path.exists(ruta_adjuntos):
            messagebox.showerror("Error", f"No existe la carpeta:\n{ruta_adjuntos}")
            return

        archivos_pdf = [
            f for f in os.listdir(ruta_adjuntos)
            if f.lower().endswith('.pdf') and not f.startswith('~')
        ]

        correos_creados = 0

        for (proveedor, fecha_pago), grupo in df.groupby([columna_proveedor, columna_fecha]):

            correo_para = grupo[columna_para].iloc[0]
            correo_cc = grupo[columna_cc].iloc[0] if columna_cc in grupo.columns else ''

            fecha_pdf = pd.to_datetime(fecha_pago).strftime('%Y%m%d')
            proveedor_key = proveedor.replace(' ', '').upper()

            adjunto_encontrado = None
            for archivo in archivos_pdf:
                archivo_key = archivo.replace(' ', '').upper()
                if archivo_key.startswith(fecha_pdf) and proveedor_key in archivo_key:
                    adjunto_encontrado = os.path.join(ruta_adjuntos, archivo)
                    break

            columnas_excluir = ['PROVEEDOR', columna_para, columna_cc]
            columnas_excluir = [c for c in columnas_excluir if c in grupo.columns]

            tabla_html = grupo.drop(columns=columnas_excluir).to_html(
                index=False, border=0, justify='center'
            )

            estilo_tabla = """
            <style>
            table {border-collapse:collapse;width:100%;font-family:Arial;font-size:13px;}
            th {background-color:#1f3a5f;color:white;padding:6px;border:1px solid #1f3a5f;text-align:center;}
            td {padding:6px;border:1px solid #1f3a5f;text-align:center;}
            </style>
            """

            cuerpo_html = f"""
            <html>
            <head>{estilo_tabla}</head>
            <body style="font-family:Arial;font-size:13px;">
            <p>Buenos días,</p>
            <p>Adjuntamos comprobante de pago correspondiente a las siguientes facturas:</p>
            {tabla_html}
            <br>
            <p>Ante cualquier duda, quedo a disposición.</p>
            <p>Buen resto de día,</p>
            <p>Saludos cordiales,</p>
            <p>
            <b>VICTOR DOMINGUEZ</b><br>
            Nutrex INC<br>
            tesoreria2@nutrexinc.com
            </p>
            </body>
            </html>
            """

            mail = outlook.CreateItem(0)
            mail.To = correo_para
            if correo_cc:
                mail.CC = correo_cc
            mail.Subject = f"COMPROBANTE DE PAGO {pd.to_datetime(fecha_pago).strftime('%d/%m/%Y')} - {proveedor}"
            mail.HTMLBody = cuerpo_html

            if adjunto_encontrado:
                mail.Attachments.Add(adjunto_encontrado)
                print(f"📎 PDF encontrado → {proveedor}")
            else:
                print(f"⚠️ PDF NO encontrado → {proveedor}")

            mail.Save()
            correos_creados += 1
            print(f"📧 Correo creado → {proveedor}")

        messagebox.showinfo("Listo ✅", f"Se crearon {correos_creados} correo(s) en borradores de Outlook.")
        print(f"\n✅ {correos_creados} correos creados en borradores.")

    except Exception as e:
        messagebox.showerror("Error al crear correos", str(e))


def ejecutar():
    """Valida el formato de fechas y lanza el proceso."""
    fechas = entrada_fecha.get().strip()

    # Validar formato si no está vacío
    if fechas != '':
        for f in fechas.split(','):
            try:
                datetime.strptime(f.strip(), '%Y-%m-%d')
            except ValueError:
                messagebox.showwarning("Formato inválido", f"Fecha no válida: '{f.strip()}'\nUsa formato YYYY-MM-DD")
                return

    ventana.destroy()
    filtrar_y_enviar(fechas)


# =====================================================
# === VENTANA TKINTER ===
# =====================================================

ventana = tk.Tk()
ventana.title("Correos Tesorería - Nutrex INC")
ventana.geometry("480x220")
ventana.resizable(False, False)
ventana.configure(bg="#f0f0f0")

# Centrar ventana en pantalla
ventana.update_idletasks()
x = (ventana.winfo_screenwidth() // 2) - (480 // 2)
y = (ventana.winfo_screenheight() // 2) - (220 // 2)
ventana.geometry(f"+{x}+{y}")

tk.Label(
    ventana, text="📧 Generador de Correos - Tesorería",
    font=("Arial", 14, "bold"), bg="#f0f0f0"
).pack(pady=(15, 5))

tk.Label(
    ventana,
    text="Ingresa fecha(s) de pago (YYYY-MM-DD), separadas por coma.\nDeja vacío para usar la fecha más reciente.",
    font=("Arial", 10), bg="#f0f0f0", justify="center"
).pack(pady=(0, 5))

entrada_fecha = tk.Entry(ventana, font=("Arial", 12), width=35, justify="center")
entrada_fecha.pack(pady=5)
entrada_fecha.insert(0, datetime.today().strftime('%Y-%m-%d'))

tk.Button(
    ventana, text="Filtrar y Crear Correos",
    font=("Arial", 11, "bold"), bg="#1f3a5f", fg="white",
    activebackground="#2d5280", activeforeground="white",
    cursor="hand2", width=25,
    command=ejecutar
).pack(pady=15)

ventana.mainloop() """